In [ ]:
import pandas as pd
import numpy as np
import requests
from pathlib import Path

PROJECT_DIR = Path("..").resolve()
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
SRC = PROJECT_DIR / "toronto-bikeshare-data"  # all monthly CSVs (2025 + 2026)

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## loading the 2025 ridership data

In [ ]:
files_2025 = sorted(SRC.glob("bikeshare_2025_*.csv"))
print(f"Found {len(files_2025)} files for 2025")

dfs = []
for f in files_2025:
    df = pd.read_csv(f, parse_dates=["Start_Time", "End_Time"], encoding="latin-1")
    print(f"  {f.name}: {len(df):,} rows")
    dfs.append(df)

rides_2025 = pd.concat(dfs, ignore_index=True)
print(f"\n2025 total: {len(rides_2025):,} rides")

Found 10 files for 2025
  bikeshare_2025_01.csv: 202,946 rows
  bikeshare_2025_02.csv: 125,670 rows
  bikeshare_2025_03.csv: 312,178 rows
  bikeshare_2025_04.csv: 471,525 rows
  bikeshare_2025_05.csv: 699,461 rows
  bikeshare_2025_06.csv: 980,624 rows
  bikeshare_2025_07.csv: 1,192,062 rows
  bikeshare_2025_08.csv: 1,164,697 rows
  bikeshare_2025_09.csv: 1,062,001 rows
  bikeshare_2025_10.csv: 872,482 rows

2025 total: 7,083,646 rides


## loading the 2026 ridership data

In [ ]:
files_2026 = sorted(SRC.glob("bikeshare_2026_*.csv"))
print(f"Found {len(files_2026)} files for 2026")

dfs = []
for f in files_2026:
    df = pd.read_csv(f, parse_dates=["Start_Time", "End_Time"], encoding="latin-1")
    print(f"  {f.name}: {len(df):,} rows")
    dfs.append(df)

rides_2026 = pd.concat(dfs, ignore_index=True)
print(f"\n2026 total: {len(rides_2026):,} rides")

Found 2 files for 2026
  bikeshare_2026_01.csv: 137,336 rows
  bikeshare_2026_02.csv: 112,910 rows

2026 total: 250,246 rides


## cleaning the trips

In [ ]:
rides = pd.concat([rides_2025, rides_2026], ignore_index=True)
original = len(rides)

rides = rides.dropna(subset=["Start_Station_Id", "Start_Time"])  # drop rows missing station or time
rides["Trip_Duration"] = pd.to_numeric(rides["Trip_Duration"], errors="coerce")
rides = rides[(rides["Trip_Duration"] >= 60) & (rides["Trip_Duration"] <= 86400)]  # keep 1min to 24hr trips

print(f"Before: {original:,} -> After: {len(rides):,} ({len(rides)/original:.1%} kept)")
print(f"Date range: {rides['Start_Time'].min()} to {rides['Start_Time'].max()}")

Before: 7,333,892 -> After: 7,315,742 (99.8% kept)
Date range: 2025-01-01 00:01:51 to 2026-02-28 23:58:13


## aggregating to hourly pickups per station

In [ ]:
rides["hour"] = rides["Start_Time"].dt.floor("h")
rides["station_id"] = rides["Start_Station_Id"].astype(int)

hourly = rides.groupby(["station_id", "hour"]).size().reset_index(name="pickups")
print(f"Hourly records (non-zero): {len(hourly):,}")
print(f"Stations: {hourly['station_id'].nunique()}")
print(f"Date range: {hourly['hour'].min()} to {hourly['hour'].max()}")

Hourly records (non-zero): 2,389,174
Stations: 1052
Date range: 2025-01-01 00:00:00 to 2026-02-28 23:00:00


filling implicit zeros so every station has an entry for every hour

In [ ]:
all_stations = sorted(hourly["station_id"].unique())
date_range = pd.date_range(start=hourly["hour"].min(), end=hourly["hour"].max(), freq="h")

full_grid = pd.MultiIndex.from_product([all_stations, date_range], names=["station_id", "hour"])
hourly_full = hourly.set_index(["station_id", "hour"]).reindex(full_grid, fill_value=0).reset_index()

print(f"Full grid: {len(hourly_full):,} rows ({len(all_stations)} stations x {len(date_range)} hours)")
print(f"Zero-pickup hours: {(hourly_full['pickups'] == 0).mean():.1%}")

Full grid: 10,705,152 rows (1052 stations x 10176 hours)
Zero-pickup hours: 77.7%


## fetching station metadata from the toronto open data GBFS API

In [ ]:
base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
params = {"id": "bike-share-toronto"}
package = requests.get(base_url + "/api/3/action/package_show", params=params, timeout=30).json()

gbfs_resource = package["result"]["resources"][1]  # index 1 is the GBFS feed
gbfs_index = requests.get(gbfs_resource["url"], timeout=30).json()

feeds = gbfs_index["data"]["en"]["feeds"]
station_info_url = next(f["url"] for f in feeds if f["name"] == "station_information")
station_json = requests.get(station_info_url, timeout=30).json()

stations = pd.DataFrame(station_json["data"]["stations"])
print(f"Fetched {len(stations)} stations, columns: {stations.columns.tolist()}")

Fetched 1031 stations, columns: ['station_id', 'external_id', 'name', 'physical_configuration', 'lat', 'lon', 'altitude', 'address', 'capacity', 'is_charging_station', 'rental_methods', 'groups', 'obcn', 'short_name', 'nearby_distance', '_ride_code_support', 'rental_uris', 'post_code', 'is_valet_station', 'cross_street']


In [ ]:
stations["station_id"] = pd.to_numeric(stations["station_id"], errors="coerce").astype("Int64")
stations = stations.dropna(subset=["station_id"])
stations = stations[["station_id", "name", "lat", "lon", "capacity"]].copy()  # keep just what we need

print(f"Stations after cleanup: {len(stations)}")
print(f"Capacity range: {stations['capacity'].min()} to {stations['capacity'].max()}")

ride_ids = set(rides["Start_Station_Id"].dropna().astype(int).unique())
meta_ids = set(stations["station_id"].dropna().unique())
print(f"Ridership stations: {len(ride_ids)}, metadata stations: {len(meta_ids)}, matched: {len(ride_ids & meta_ids)}")

Stations after cleanup: 1031
Capacity range: 0 to 87
Ridership stations: 1052, metadata stations: 1031, matched: 999


## fetching hourly weather from open-meteo (jan 2025 to feb 2026)

In [ ]:
WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"

weather_params = {
    "latitude": 43.65,
    "longitude": -79.38,
    "start_date": "2025-01-01",
    "end_date": "2026-02-28",
    "hourly": "temperature_2m,precipitation,wind_speed_10m,weather_code",
    "timezone": "America/Toronto"
}

resp = requests.get(WEATHER_URL, params=weather_params, timeout=60)
resp.raise_for_status()
weather_json = resp.json()

weather = pd.DataFrame({
    "hour": pd.to_datetime(weather_json["hourly"]["time"]),
    "temperature_c": weather_json["hourly"]["temperature_2m"],
    "precipitation_mm": weather_json["hourly"]["precipitation"],
    "wind_speed_kmh": weather_json["hourly"]["wind_speed_10m"],
    "weather_code": weather_json["hourly"]["weather_code"]
})

print(f"Weather rows: {len(weather)}, range: {weather['hour'].min()} to {weather['hour'].max()}")
print(f"Null counts: {weather.isnull().sum().to_dict()}")

Weather rows: 10176, range: 2025-01-01 00:00:00 to 2026-02-28 23:00:00
Null counts: {'hour': 0, 'temperature_c': 0, 'precipitation_mm': 0, 'wind_speed_kmh': 0, 'weather_code': 0}


In [ ]:
weather["temperature_c"] = weather["temperature_c"].interpolate(method="linear")  # fill any gaps
weather["precipitation_mm"] = weather["precipitation_mm"].fillna(0)
weather["wind_speed_kmh"] = weather["wind_speed_kmh"].interpolate(method="linear")
weather["weather_code"] = weather["weather_code"].ffill()

print(f"Nulls after cleaning: {weather.isnull().sum().to_dict()}")

Nulls after cleaning: {'hour': 0, 'temperature_c': 0, 'precipitation_mm': 0, 'wind_speed_kmh': 0, 'weather_code': 0}


## saving

In [ ]:
rides.to_csv(RAW_DIR / "rides_raw.csv", index=False)
stations.to_csv(RAW_DIR / "stations.csv", index=False)
hourly_full.to_csv(PROCESSED_DIR / "hourly_pickups.csv", index=False)
weather.to_csv(PROCESSED_DIR / "weather_hourly.csv", index=False)

print("data/raw:")
for f in sorted(RAW_DIR.iterdir()):
    print(f"  {f.name}, {f.stat().st_size/1e6:.1f} MB")
print("\ndata/processed:")
for f in sorted(PROCESSED_DIR.iterdir()):
    print(f"  {f.name}, {f.stat().st_size/1e6:.1f} MB")

data/raw:
  rides_raw.csv, 1215.5 MB
  stations.csv, 0.1 MB
  weather_hourly.csv, 0.3 MB

data/processed:
  hourly_pickups.csv, 299.9 MB
  stations_clean.csv, 0.1 MB
  test.csv, 2.2 MB
  train.csv, 6.9 MB
  weather_clean.csv, 0.3 MB
  weather_hourly.csv, 0.4 MB
